# Lab 2 · Tools and the agent loop

In Lab 1 the LLM read documents and answered, but it could not act. In this lab the LLM gets **tools**: functions that open a PDF, recalculate a total, look up a purchase order in a database and save a result. It decides for itself which tool to call and when.

**The task:** check 8 PDF invoices against the purchase orders (POs) in a database, and record whether each is approved or rejected, with a reason.

| Part | What you do | Time |
|---|---|---|
| 1 | A tool is a function with a description | 10 min |
| 2 | One tool call by hand | 10 min |
| 3 | Write the agent loop yourself | 15 min |
| 4 | Check all 8 invoices and measure | 10 min |
| 5 | Tool design and error handling | 10 min |
| 6 | `create_agent`, and comparing with a workflow | 10 min |
| 7 | Stretch: serve the tools over MCP | If time allows |

Look for `# TODO` in the code. Every cell runs before you change it. Before you start, save this file as `lab2_tool_agent.mine.ipynb`.

## 1. A tool is a function with a description (10 min)

This lab's tools live in `agentlab/invoices.py`. Each is an ordinary function wrapped with `@tool`. We can call them directly like any function, through `.invoke()`.

In [ ]:
import json
import time

import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

from agentlab import get_llm, release_model
from agentlab.invoices import (INVOICE_DIR, SYSTEM_PROMPT, TOOLS, check_invoice_total, compare_with_po,
                               create_database, evaluate_checks, read_invoice, record_result)

create_database(reset=True)  # a fresh database: purchase orders, and no results yet
llm = get_llm()

print(read_invoice.invoke({"filename": "inv_04.pdf"}))
print()
print(check_invoice_total.invoke({"filename": "inv_04.pdf"}))

The model cannot see a tool's code. It sees only the **name, description and arguments** that LangChain builds from the function. The cell below shows exactly what the model receives.

In [ ]:
for t in TOOLS:
    spec = convert_to_openai_tool(t)["function"]
    print(f"{spec['name']}({', '.join(spec['parameters']['properties'])})")
    print(f"    {spec['description']}\n")

print(json.dumps(convert_to_openai_tool(record_result), indent=2))

**Your job:** write a tool `list_invoices` that returns the names of all invoice files in `INVOICE_DIR` as a JSON list. Write a docstring that tells the model what the tool does and when to use it, then look at the schema it produces.

In [ ]:
@tool
def list_invoices() -> str:
    """TODO: one or two sentences for the model: what this returns and when to use it."""
    # TODO: return json.dumps of the sorted file names (p.name) of INVOICE_DIR.glob("*.pdf")
    return "[]"


print(list_invoices.invoke({}))
print(convert_to_openai_tool(list_invoices)["function"])

## 2. One tool call by hand (10 min)

`bind_tools()` attaches the tool descriptions to every request. When the model wants a tool it **does not answer in text**. It answers with `tool_calls`: the tool name and the arguments it wants.

In [ ]:
DEMO_SYSTEM = "You answer questions about invoices using the tools. Never do arithmetic yourself."
model = llm.bind_tools(TOOLS)
messages = [SystemMessage(DEMO_SYSTEM), HumanMessage("Is the total on inv_04.pdf correct?")]

reply = model.invoke(messages)
print("text:      ", repr(reply.text))
print("tool_calls:", reply.tool_calls)

The model has only **asked** for a tool; nothing has run yet. Now we act as the harness: run the tool ourselves, send the result back as a `ToolMessage` carrying the same `tool_call_id`, and ask the model again.

In [ ]:
tools_by_name = {t.name: t for t in TOOLS}

call = reply.tool_calls[0]
result = tools_by_name[call["name"]].invoke(call["args"])
print("tool result:", result)

messages += [reply, ToolMessage(content=result, tool_call_id=call["id"])]
final = model.invoke(messages)
print("\nfinal answer:", final.text)
print("\nmessages so far:", [type(m).__name__ for m in messages + [final]])

## 3. Write the agent loop yourself (15 min)

Part 2 was one round. Real work takes several: the model calls a tool, we send the result back, the model calls the next tool, until it answers without calling another. That is the agent loop.

**Your job:** fill in `TODO 1` and `TODO 2` in `run_agent`. `run_tool` is kept separate because part 5 makes it safer.

In [ ]:
def run_tool(call, tools):
    """Run one tool call the model asked for and return the result as text."""
    return {t.name: t for t in tools}[call["name"]].invoke(call["args"])


def run_agent(task, tools=TOOLS, system=SYSTEM_PROMPT, max_steps=8):
    """Call the model, run the tools it asks for, and repeat until it stops asking. Returns (answer, messages)."""
    model = llm.bind_tools(tools)
    messages = [SystemMessage(system), HumanMessage(task)]
    for step in range(max_steps):
        reply = model.invoke(messages)
        messages.append(reply)
        # TODO 1: if reply.tool_calls is empty, the agent has finished: return reply.text, messages
        # TODO 2: otherwise, for each call in reply.tool_calls, append
        #         ToolMessage(content=run_tool(call, tools), tool_call_id=call["id"])
    return "(stopped: max_steps reached)", messages


def show_trace(messages):
    """Print what happened at each step of a run."""
    for m in messages:
        if isinstance(m, HumanMessage):
            print(f"TASK   {m.content}")
        elif getattr(m, "tool_calls", None):
            for c in m.tool_calls:
                print(f"CALL   {c['name']}({json.dumps(c['args'], ensure_ascii=False)})")
        elif isinstance(m, ToolMessage):
            print(f"RESULT {m.text[:110]}")
        elif m.type == "ai":
            print(f"ANSWER {m.text}")


answer, messages = run_agent("Check invoice inv_03.pdf.")
show_trace(messages)

## 4. Check all 8 invoices and measure (10 min)

Run the agent on every invoice, then compare what `record_result` saved with the answer key in `data/invoices_labels.csv`. A rejection only counts as correct if its reason names the check that actually failed. This takes about 1–3 minutes.

In [ ]:
def run_all(runner):
    """Check every invoice with runner(task) -> (answer, messages). Returns per-invoice steps and time."""
    create_database(reset=True)
    rows = []
    for path in sorted(INVOICE_DIR.glob("*.pdf")):
        start = time.time()
        answer, messages = runner(f"Check invoice {path.name}.")
        calls = [c["name"] for m in messages for c in (getattr(m, "tool_calls", None) or [])]
        rows.append({"file": path.name, "tool_calls": len(calls), "llm_calls": sum(m.type == "ai" for m in messages),
                     "seconds": round(time.time() - start, 1), "answer": answer[:60]})
    release_model()
    return pd.DataFrame(rows).set_index("file")


runs = run_all(run_agent)
results = evaluate_checks().join(runs)
print(f"correct: {results['correct'].sum()} / {len(results)}   LLM calls: {results['llm_calls'].sum()}   seconds: {results['seconds'].sum():.0f}")
results[["expected_verdict", "issue", "verdict", "reason", "correct", "tool_calls"]]

## 5. Tool design and error handling (10 min)

### 5.1 A badly designed tool
The tools in `agentlab` were designed with care. The one below does the same job as `compare_with_po`, but its name is vague, its description is one word, and it makes the model **copy the PO number itself**. Run it on 3 invoices and look in the trace at the arguments the model sends.

In [ ]:
from agentlab.invoices import query


@tool
def compare(po: str) -> str:
    """Compare."""
    rows = query("SELECT * FROM po_lines WHERE po_number = ?", (po,))
    return json.dumps(rows) if rows else "[]"


bad_tools = [read_invoice, check_invoice_total, compare, record_result]
bad_system = SYSTEM_PROMPT.replace("compare_with_po", "compare")
create_database(reset=True)
for name in ["inv_03.pdf", "inv_06.pdf", "inv_08.pdf"]:
    print(f"===== {name}")
    answer, messages = run_agent(f"Check invoice {name}.", tools=bad_tools, system=bad_system)
    show_trace(messages[2:])
release_model()
evaluate_checks().loc[["inv_03.pdf", "inv_06.pdf", "inv_08.pdf"], ["expected_verdict", "issue", "verdict", "reason", "correct"]]

Look at the `po` argument the model sends. Did it open the invoice and read it, or make a number up? And what happened to `inv_08.pdf`, which has no PO number at all? Mistakes like this raise no error. The system looks as if it is working.

The most effective fix is in the tool design, not the prompt. `compare_with_po` in `agentlab` finds the PO number on the invoice itself. The model passes only the file name, so it has no chance to invent a number.

### 5.2 When a tool raises an error
A tool we write ourselves can raise an exception, for example when the model passes a file name that does not exist, or calls a tool name that does not exist. At the moment `run_tool` lets the exception escape, and the whole loop crashes.

**Your job:** change `run_tool` so it catches the exception and returns the error to the model as JSON, and answers politely when the model calls a tool that does not exist. Then run the cell, which uses a tool that raises when the file is missing.

In [ ]:
def run_tool(call, tools):
    """Run one tool call the model asked for and return the result as text."""
    by_name = {t.name: t for t in tools}
    # TODO: if call["name"] is not in by_name, return a JSON error naming the tools that do exist
    # TODO: wrap the call in try/except Exception and return json.dumps({"error": str(e)}) on failure
    return by_name[call["name"]].invoke(call["args"])


@tool
def strict_total(filename: str) -> str:
    """Recalculate an invoice total. Raises an error if the file does not exist."""
    path = INVOICE_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"{filename} not found; available files are inv_01.pdf to inv_08.pdf")
    return check_invoice_total.invoke({"filename": filename})


try:
    answer, messages = run_agent("Is the total on invoice_9.pdf correct?", tools=[strict_total], system=DEMO_SYSTEM)
    show_trace(messages)
except Exception as error:
    print(f"The loop crashed: {type(error).__name__}: {error}")

## 6. `create_agent`, and comparing with a workflow (10 min)

### 6.1 `create_agent`
LangChain has a ready-made agent loop called `create_agent`. It does what your `run_agent` does, plus things we use in later modules, such as memory and pausing for a person's approval. Inside, it is a LangGraph graph, which we study in Module 3.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(llm, TOOLS, system_prompt=SYSTEM_PROMPT)
out = agent.invoke({"messages": [HumanMessage("Check invoice inv_07.pdf.")]})
show_trace(out["messages"])


def run_with_create_agent(task):
    out = agent.invoke({"messages": [HumanMessage(task)]})
    return out["messages"][-1].text, out["messages"]


agent_runs = run_all(run_with_create_agent)
agent_results = evaluate_checks().join(agent_runs)
print(f"\ncreate_agent: correct {agent_results['correct'].sum()} / 8, LLM calls {agent_results['llm_calls'].sum()}, {agent_results['seconds'].sum():.0f}s")

### 6.2 Does this job really need an agent?
Every invoice goes through the same steps in the same order. By Module 1's reasoning, a job whose steps can be stated in advance should be a **workflow**. The function below calls the same tools in a fixed order, with no LLM at all.

In [ ]:
def check_with_workflow(filename):
    """The same checks in a fixed order, decided by rules: no LLM."""
    total = json.loads(check_invoice_total.invoke({"filename": filename}))
    po = json.loads(compare_with_po.invoke({"filename": filename}))
    if "error" in total or not total["total_ok"]:
        verdict, reason = "reject", "total"
    elif "error" in po:
        verdict, reason = "reject", "no_po" if "no PO number" in po["error"] else "po_not_found"
    elif not po["matches_po"]:
        verdict, reason = "reject", "unit_price" if "unit price" in " ".join(po["problems"]) else "quantity"
    else:
        verdict, reason = "approve", "all checks passed"
    record_result.invoke({"filename": filename, "verdict": verdict, "reason": reason})


create_database(reset=True)
start = time.time()
for path in sorted(INVOICE_DIR.glob("*.pdf")):
    check_with_workflow(path.name)
workflow_seconds = time.time() - start

pd.DataFrame({
    "correct (of 8)": [results["correct"].sum(), agent_results["correct"].sum(), evaluate_checks()["correct"].sum()],
    "LLM calls": [results["llm_calls"].sum(), agent_results["llm_calls"].sum(), 0],
    "seconds": [round(results["seconds"].sum()), round(agent_results["seconds"].sum()), round(workflow_seconds, 1)],
}, index=["your run_agent", "create_agent", "fixed workflow"])

The workflow does as well or better, with no LLM calls and far faster. That does not make agents useless; it means **this job** does not need one. We used it to learn how tools and the loop work because the results are clear and measurable.

An agent pays off when the steps depend on what is found: an invoice that refers to several purchase orders, a case that needs a goods receipt or credit note opened as well, or a user question where nobody can say in advance which tool it needs.

## 7. Stretch: serve the tools over MCP

At the moment the tools work only inside our own Python. **MCP (Model Context Protocol)** is a standard that lets any application discover and call our tools. `mcp_server.py` in this folder serves all 4 tools over MCP. The cell below starts that server as a separate process, fetches the tool list through MCP, and hands it to an agent.

Question: compare the schema of `compare_with_po` received over MCP with the one from part 1. What is missing, and how could that affect the model?

In [ ]:
import sys

from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "invoices": {"command": sys.executable, "args": [str(INVOICE_DIR.parent.parent / "lab-02-tool-use" / "mcp_server.py")], "transport": "stdio"},
})
mcp_tools = await client.get_tools()
for t in mcp_tools:
    print(t.name, "->", t.args_schema)

mcp_agent = create_agent(llm, mcp_tools, system_prompt=SYSTEM_PROMPT)
create_database(reset=True)
out = await mcp_agent.ainvoke({"messages": [HumanMessage("Check invoice inv_08.pdf.")]})
release_model()
show_trace(out["messages"])

## Wrap-up and what to hand in

Answer these 4 questions in 2–3 sentences each, and hand them in with a screenshot of the comparison table from part 6.2.

1. In part 2, who actually ran `check_invoice_total`: the model or your code? Why does that matter for safety?
2. In part 5.1, what kind of mistake did the `compare` tool lead the model into? Why does fixing the tool work better than writing a more detailed prompt?
3. Why should a tool return an error message to the model instead of raising an exception?
4. From the table in part 6.2, what kind of work in your job calls for an agent, and what kind is better as a workflow?

**From notebook to program:** open `invoice_agent.py` in the same folder. It is this agent written to be run for real. `README.md` shows how to run it and `invoice_agent.en.md` explains it part by part.

**Project milestone 2:** list the 3–5 tools your group's agent needs. Write the name, description and arguments of each following the principles in part 5, and say which ones have side effects.